In [27]:
#!pip install transformers accelerate

In [28]:
#!pip install --upgrade ipywidgets jupyterlab_widgets tqdm rich #rich progress bar

In [29]:
import os
#for import, set to "0", for static training without pushing back to HuggingFace, set to "1"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
access_token = #"YOUR_TOKEN_HERE"

In [30]:
from tqdm.rich import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# Simonlee711/Clinical_ModernBERT
note that performs optimally with GPUs with flash attention (!pip install flash-attn), but do not want to train on more than V100s in Verily due to cost.
https://huggingface.co/Simonlee711/Clinical_ModernBERT

In [32]:
model_id = "Simonlee711/Clinical_ModernBERT"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
model = AutoModel.from_pretrained(model_id, 
                                  token=access_token,
                                  attn_implementation="eager", 
                                  torch_dtype=torch.bfloat16,  # Solves the float32 error
                                  device_map=device)

Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Simonlee711/Clinical_ModernBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
text = "Please explain text-[MASK] regression."
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
print("Last hidden state shape:", last_hidden_state.shape)
# embedding of the very last token in the sequence:
last_token_hidden_state = last_hidden_state[:, 0, :] #class
print("Last token hidden state shape:", last_token_hidden_state.shape)
#print(last_token_hidden_state)

Last hidden state shape: torch.Size([1, 9, 768])
Last token hidden state shape: torch.Size([1, 768])


# BioClinical-ModernBERT
note that performs optimally with GPUs with flash attention (!pip install flash-attn), but do not want to train on more than V100s in Verily due to cost.
https://huggingface.co/thomas-sounack/BioClinical-ModernBERT-base

In [34]:
model_id = "thomas-sounack/BioClinical-ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, 
                                          token=access_token,
                                          device_map=device)
model = AutoModel.from_pretrained(model_id, 
                                  token=access_token,
                                  attn_implementation="eager", 
                                  torch_dtype=torch.bfloat16,  # Solves the float32 error
                                  device_map=device)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: thomas-sounack/BioClinical-ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
text = "Please explain text-[MASK] regression."
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
print("Last hidden state shape:", last_hidden_state.shape)
# embedding of the very last token in the sequence:
bio_last_token_hidden_state = last_hidden_state[:, 0, :] #class
print("Last token hidden state shape:", bio_last_token_hidden_state.shape)
#print(last_token_hidden_state)

Last hidden state shape: torch.Size([1, 9, 768])
Last token hidden state shape: torch.Size([1, 768])


# Gemma
google/gemma-4-E2B (Effective 2B size optimized for edge devices) 

https://huggingface.co/google/gemma-4-12B

smallest Gemma

https://huggingface.co/google/gemma-3-270m

In [36]:
%%time
model_id = "google/gemma-3-270m" #"google/gemma-4-E2B-it"
tokenizer = AutoTokenizer.from_pretrained(model_id, 
                                          token=access_token)
model = AutoModelForCausalLM.from_pretrained(model_id, 
                                             token=access_token,
                                             ignore_mismatched_sizes=True, 
                                             device_map=device) #device_map="auto", )

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

CPU times: user 1.96 s, sys: 231 ms, total: 2.19 s
Wall time: 2.43 s


In [37]:
%%time
prompt = "Explain text regression"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

CPU times: user 293 ms, sys: 29.3 ms, total: 322 ms
Wall time: 165 ms


In [38]:
all_hidden_states = outputs.hidden_states
last_hidden_state = all_hidden_states[-1]
print("Last hidden state shape:", last_hidden_state.shape)
# embedding of the very last token in the sequence:
last_token_hidden_state = last_hidden_state[:, -1, :]
print("Last token hidden state shape:", last_token_hidden_state.shape)
print(last_token_hidden_state)

Last hidden state shape: torch.Size([1, 4, 640])
Last token hidden state shape: torch.Size([1, 640])
tensor([[ 4.1875e+00, -1.7031e+00,  1.1406e+00,  4.7812e+00, -5.6562e+00,
          2.9844e+00, -6.0938e-01, -8.9375e+00,  8.0078e-01,  3.4844e+00,
         -1.7031e+00,  1.2812e+00, -6.3125e+00,  2.9688e-01, -1.7578e+00,
         -2.4219e+00, -3.2812e+00, -1.6328e+00,  2.2500e+00, -1.3203e+00,
         -6.8848e-02, -1.4297e+00, -4.5312e+00,  2.3438e+00,  1.9297e+00,
         -1.9297e+00, -2.2812e+00, -2.0625e+00,  2.2031e+00,  5.7188e+00,
          1.4219e+00,  4.0000e+00,  4.2188e-01, -3.0781e+00, -1.2207e-01,
          2.3926e-01, -1.4062e+00,  1.4766e+00,  8.7891e-01, -2.6406e+00,
          9.7500e+00,  1.3086e-01,  4.3555e-01,  1.4688e+00,  2.0000e+00,
          2.3750e+00,  6.0625e+00, -2.0938e+00, -1.7500e+00, -1.5703e+00,
          5.2812e+00,  4.0771e-02,  4.4375e+00, -2.2031e+00,  1.9141e+00,
          1.9375e+00, -5.1172e-01, -2.8750e+00, -1.7266e+00, -4.2812e+00,
          1

# medicalai/ClinicalBERT
more oriented toward training on clinical free text EHR data (not as much in scientific papers) https://huggingface.co/medicalai/ClinicalBERT

In [39]:
model_id = "medicalai/ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
model = AutoModel.from_pretrained(model_id, 
                                  token=access_token, 
                                  device_map=device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
text = "Please explain text-[MASK] regression."
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
print("Last hidden state shape:", last_hidden_state.shape)
# embedding of the very last token in the sequence:
last_token_hidden_state = last_hidden_state[:, 0, :] #class
print("Last token hidden state shape:", last_token_hidden_state.shape)
#print(last_token_hidden_state)

Last hidden state shape: torch.Size([1, 12, 768])
Last token hidden state shape: torch.Size([1, 768])


# "emilyalsentzer/Bio_ClinicalBERT" 
ClinicalBERT extended to MIMIC III notes https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT


In [41]:
model_id = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
model = AutoModel.from_pretrained(model_id, 
                                  token=access_token,
                                  device_map=device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
text = "Please explain text-[MASK] regression."
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
print("Last hidden state shape:", last_hidden_state.shape)
# embedding of the very last token in the sequence:
last_token_hidden_state = last_hidden_state[:, 0, :] #class
print("Last token hidden state shape:", last_token_hidden_state.shape)
#print(last_token_hidden_state)

Last hidden state shape: torch.Size([1, 10, 768])
Last token hidden state shape: torch.Size([1, 768])


# HRM
1 B-parameter language model checkpoint built on the Hierarchical Reasoning Model (HRM) architecture, trained by Sapient Intelligence from scratch on structured public datasets.
https://huggingface.co/sapientinc/HRM-Text-1B

In [43]:
# %%time
# model_id = "sapientinc/HRM-Text-1B"
# tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
# model = AutoModelForCausalLM.from_pretrained(model_id, token=access_token)

In [44]:
# # synth,cot composite — reasoning / CoT style (see Disclaimer for other modes)
# condition = "<|quad_end|><|object_ref_end|>"
# prompt = f"<|im_start|>{condition}Explain text regression.<|im_end|>"

# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# # Mark the prompt as a single bidirectional prefix block — see "PrefixLM mask" below.
# inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

In [45]:
# %%time
# with torch.no_grad():
#     out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
# print(tokenizer.decode(out[0], skip_special_tokens=False))

In [46]:
# %%time
# with torch.no_grad():
#     out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
# print(tokenizer.decode(out[0], skip_special_tokens=False))

In [47]:
# %%time
# with torch.no_grad():
#     out = model.generate(**inputs, max_new_tokens=256, do_sample=True)
# print(tokenizer.decode(out[0], skip_special_tokens=False))

In [48]:
# %%time
# with torch.no_grad():
#     outputs = model(**inputs, output_hidden_states=True)

In [49]:
# all_hidden_states = outputs.hidden_states
# last_hidden_state = all_hidden_states[-1]
# print("Last hidden state shape:", last_hidden_state.shape)
# # embedding of the very last token in the sequence:
# last_token_hidden_state = last_hidden_state[:, -1, :]
# print("Last token hidden state shape:", last_token_hidden_state.shape)
# print(last_token_hidden_state)

# MedGemma
google/medgemma-4b-pt	2025-05-20	GA	LLM for medical image and text comprehension pre-trained
https://console.cloud.google.com/agent-platform/publishers/google/model-garden/medgemma

In [50]:
# %%time
# model_id = "google/medgemma-4b-pt"
# tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", token=access_token)

In [51]:
# %%time
# prompt = "Explain text regression"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# with torch.no_grad():
#     outputs = model(**inputs, output_hidden_states=True)

In [52]:
# all_hidden_states = outputs.hidden_states
# last_hidden_state = all_hidden_states[-1]
# print("Last hidden state shape:", last_hidden_state.shape)
# # embedding of the very last token in the sequence:
# last_token_hidden_state = last_hidden_state[:, -1, :]
# print("Last token hidden state shape:", last_token_hidden_state.shape)
# print(last_token_hidden_state)